# Memory Retrieval — Finding the Right Memories at Scale

Based on:
- [Zep: Temporal Knowledge Graph for Agent Memory](https://arxiv.org/abs/2501.13956) — Rasmussen et al., 2025
- [PersonaAgent with GraphRAG](https://arxiv.org/abs/2511.17467) — Liang et al., 2025
- [HippoRAG 2: From RAG to Memory](https://arxiv.org/abs/2502.14802) — Jimenez Gutierrez et al., 2025

## The Problem

When core memory grows to dozens of sections (persona, travel preferences, food preferences, work schedule, past trips, loyalty programs, communication style, emergency contacts...), dumping everything into the LLM context is:

- **Expensive** — thousands of irrelevant tokens per query
- **Noisy** — the LLM must filter signal from noise
- **Degrading** — response quality drops with irrelevant context

## The Solution: Smart Retrieval

| Strategy | How it works | Precision | Cost |
|----------|-------------|-----------|------|
| Dump all | Load entire memory | Low | High |
| Keyword | Exact string match | Medium | Medium |
| Semantic | Embedding similarity, top-k | High | Low |

## What We Test

A user with 8 memory sections asks: *"What food do I like and what should I avoid?"*

Only `food_preferences` (and maybe `past_trips`) are relevant. Why load all 8 sections?

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any supported model provider. Change the model in the setup cell below.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    'OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

In [ ]:
import json, time, os

os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent
# OpenAI-compatible model interface
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from tools import seed_memory, memory_dump_all, memory_search_keyword, memory_search_semantic

load_dotenv()

MODEL = OpenAIModel(model_id='gpt-4o-mini')

QUERY = 'I need restaurant recommendations for my upcoming trip. What food do I like and what should I avoid?'


def count_context_tokens(agent) -> int:
    """Estimate tokens from all messages in conversation history."""
    total = 0
    for msg in agent.messages:
        content = msg.get('content', [])
        if isinstance(content, str):
            total += len(content) // 4
        elif isinstance(content, list):
            for block in content:
                if isinstance(block, dict):
                    if 'text' in block:
                        total += len(block['text']) // 4
                    elif 'toolResult' in block:
                        for item in block['toolResult'].get('content', []):
                            if 'text' in item:
                                total += len(item['text']) // 4
                    elif 'toolUse' in block:
                        total += len(json.dumps(block['toolUse'].get('input', {}))) // 4
    return total


print('Setup complete!')

---
## The Memory Profile

We seed the agent with 8 memory sections representing a frequent traveler:

| Section | Content |
|---------|---------|
| `persona` | Name, role, location, languages |
| `travel_preferences` | Style, stars, budget, amenities |
| `food_preferences` | Dietary (vegetarian), allergies (shellfish), favorites |
| `work_schedule` | Timezone, meeting days, blackout dates |
| `past_trips` | 6 past trips with ratings and notes |
| `loyalty_programs` | Hotel/airline status, miles |
| `communication_style` | Tone, format, language |
| `emergency_contacts` | Primary, secondary, insurance |

---
## Test 1 — Dump All Memory (Baseline)

`memory_dump_all` loads the entire memory into context. All 8 sections, every field.

For a food question, the agent receives emergency contacts, work schedule, loyalty programs — all irrelevant noise.

**Expected:** High token count. Answer is correct but inefficient.

In [ ]:
agent_dump = Agent(
    model=MODEL,
    system_prompt='You are a personal assistant. Use memory_dump_all to retrieve the user\'s memory, then answer. Be concise.',
    tools=[memory_dump_all],
)
sections = seed_memory(agent_dump)
print(f'Seeded {sections} memory sections')

print(f'\nQuery: {QUERY}\n')
start = time.time()
agent_dump(QUERY)
time_dump = time.time() - start
tokens_dump = count_context_tokens(agent_dump)

full_size = len(json.dumps(agent_dump.state.get('core_memory') or {}))
print(f'\nTime: {time_dump:.1f}s | Tokens: {tokens_dump:,} | Memory loaded: {full_size:,} bytes (ALL)')

---
## Test 2 — Keyword Search

`memory_search_keyword` filters sections by exact string match. The agent searches for "food" or "restaurant".

**Limitation:** If the user asks about "diet" but the section uses "dietary", keyword search might work. But asking about "what I eat" won't match "food_preferences".

**Expected:** Fewer tokens than dump-all, but misses some relevant context.

In [ ]:
agent_keyword = Agent(
    model=MODEL,
    system_prompt='You are a personal assistant. Use memory_search_keyword to find relevant memories, then answer. Be concise.',
    tools=[memory_search_keyword],
)
seed_memory(agent_keyword)

print(f'Query: {QUERY}\n')
start = time.time()
agent_keyword(QUERY)
time_kw = time.time() - start
tokens_kw = count_context_tokens(agent_keyword)

print(f'\nTime: {time_kw:.1f}s | Tokens: {tokens_kw:,} | Strategy: keyword')

---
## Test 3 — Semantic Search (top-3)

`memory_search_semantic` converts the query and each memory section into embeddings, then ranks by cosine similarity. Returns only the top-3 most relevant sections.

For a food question, it should surface `food_preferences` and `past_trips` (which contain restaurant notes) — and skip `work_schedule`, `emergency_contacts`, etc.

**Expected:** Lowest token count with highest precision.

In [ ]:
agent_semantic = Agent(
    model=MODEL,
    system_prompt='You are a personal assistant. Use memory_search_semantic to find relevant memories, then answer. Be concise.',
    tools=[memory_search_semantic],
)
seed_memory(agent_semantic)

print(f'Query: {QUERY}\n')
start = time.time()
agent_semantic(QUERY)
time_sem = time.time() - start
tokens_sem = count_context_tokens(agent_semantic)

print(f'\nTime: {time_sem:.1f}s | Tokens: {tokens_sem:,} | Strategy: semantic (top-3)')

---
## Test 4 — Multi-Turn with Semantic Retrieval

Three different questions, each retrieving different memory sections:

| Turn | Query | Expected sections |
|------|-------|-------------------|
| 1 | Restaurant recommendations | food_preferences, past_trips |
| 2 | Work schedule conflicts | work_schedule |
| 3 | Airline miles for Zurich | loyalty_programs, travel_preferences |

In [ ]:
agent_multi = Agent(
    model=MODEL,
    system_prompt='You are a personal travel assistant. Use memory_search_semantic to find relevant context before answering. Be concise.',
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[memory_search_semantic, memory_search_keyword],
)
seed_memory(agent_multi)

queries = [
    'What restaurants should I look for on my Zurich trip?',
    'When should I avoid scheduling this trip based on my work calendar?',
    'Do I have enough miles to fly United to Zurich?',
]

for i, q in enumerate(queries, 1):
    print(f'\nTurn {i}: {q}')
    agent_multi(q)
    print(f'  Tokens after turn {i}: {count_context_tokens(agent_multi):,}')

---
## Comparison

In [ ]:
print(f"{'Strategy':<40} {'Tokens':>10} {'Time':>8} {'Precision':>10}")
print('-' * 70)
print(f"{'Test 1 — Dump all memory':<40} {tokens_dump:>10,} {time_dump:>6.1f}s {'Low':>10}")
print(f"{'Test 2 — Keyword search':<40} {tokens_kw:>10,} {time_kw:>6.1f}s {'Medium':>10}")
print(f"{'Test 3 — Semantic search (top-3)':<40} {tokens_sem:>10,} {time_sem:>6.1f}s {'High':>10}")

if tokens_dump > tokens_sem > 0:
    reduction = (1 - tokens_sem / tokens_dump) * 100
    print(f'\nSemantic search uses {reduction:.0f}% fewer tokens than dump-all')

---
## Summary

### When to Use Each Strategy

- **Dump all** — Only when memory is small (<5 sections) and all context might be relevant
- **Keyword** — When you know the exact terms (section names, specific values)
- **Semantic** — Default choice for production. Scales to hundreds of memory sections

### Production Upgrades

| Component | This demo | Production |
|-----------|-----------|------------|
| Embeddings | Bag-of-words | OpenAI `text-embedding-3-small` |
| Index | Linear scan | FAISS or pgvector |
| Temporal | None | Zep-style recency weighting |
| Graph | None | PersonaAgent knowledge graph |

## References

- [Zep: Temporal Knowledge Graph](https://arxiv.org/abs/2501.13956) — 94.8% on DMR, 90% less latency
- [PersonaAgent with GraphRAG](https://arxiv.org/abs/2511.17467) — +56.1% F1 on movie tagging
- [HippoRAG 2](https://arxiv.org/abs/2502.14802) — +7% associative memory
- [Strands Agent State](https://github.com/strands-agents/sdk-python)
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)